In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

CATALOG     = "hospital"
LANDING     = f"/Volumes/{CATALOG}/bronze/landing"
CHECKPOINTS = f"/Volumes/{CATALOG}/bronze/checkpoints/bronze"

SOURCES = {
    "patients":  ("patient_raw",   ["patient_id", "first_name", "last_name", "gender", "dob", "city"]),
    "hospitals": ("hospital_raw",  ["hospital_id", "hospital_name", "city", "bed_count"]),
    "diagnosis": ("diagnosis_raw", ["diagnosis_code", "diagnosis_desc"]),
    "visits":    ("visit_raw",     ["visit_id", "patient_id", "hospital_id", "admission_date",
                                    "discharge_date", "diagnosis_code", "cost"]),
}

In [0]:
def ingest_bronze(folder: str, table: str, columns: list[str]) -> None:
    schema = StructType([StructField(c, StringType(), True) for c in columns])

    df = (spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("rescuedDataColumn", "_rescued_data")
            .schema(schema)
            .load(f"{LANDING}/{folder}/")
            .withColumn("_source_file",      F.col("_metadata.file_path"))
            .withColumn("_file_modified_at", F.col("_metadata.file_modification_time"))
            .withColumn("_ingested_at",      F.current_timestamp()))

    (df.writeStream
       .option("checkpointLocation", f"{CHECKPOINTS}/{table}/checkpoint")
       .trigger(availableNow=True)
       .toTable(f"{CATALOG}.bronze.{table}")
       .awaitTermination())
    print(f"✅ {CATALOG}.bronze.{table}")

In [0]:
for folder, (table, columns) in SOURCES.items():
    ingest_bronze(folder, table, columns)

In [0]:
%sql
SELECT 'patient_raw' AS t, COUNT(*) AS n FROM hospital.bronze.patient_raw
UNION ALL SELECT 'hospital_raw',  COUNT(*) FROM hospital.bronze.hospital_raw
UNION ALL SELECT 'diagnosis_raw', COUNT(*) FROM hospital.bronze.diagnosis_raw
UNION ALL SELECT 'visit_raw',     COUNT(*) FROM hospital.bronze.visit_raw

In [0]:
%sql
SELECT visit_id, cost, _source_file, _ingested_at, _rescued_data
FROM hospital.bronze.visit_raw
ORDER BY visit_id